In [ ]:
import logging, sys
import pandas as pd
import hiring_compass_au.data.storage.db as db
import hiring_compass_au.workspace as workspace

%matplotlib inline
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True,   # important dans notebooks (reconfigure)
)

pd.set_option("display.max.columns", None)
pd.options.display.float_format = '{:,.0f}'.format
ws = workspace.get_workspace()

# Emails

In [ ]:
with db.get_connection(ws.db_path) as conn:
    emails = pd.read_sql_query(
        "SELECT * FROM emails",
        conn,
    )
    
emails.head()

In [ ]:
emails.info()

In [ ]:
for c in ["received_at","indexed_at","fetched_at","parsed_at"]:
    emails[c] = pd.to_datetime(emails[c])

In [ ]:
emails.info()

In [ ]:
emails.describe()

# Job Hits

In [ ]:
with db.get_connection(ws.db_path) as conn:
    job_hits = pd.read_sql_query(
        "SELECT * FROM email_job_hits",
        conn,
    )

job_hits.head()

In [ ]:
job_hits.info()

In [ ]:
job_hits.describe()

### Hit confidence

In [ ]:
job_hits[job_hits["hit_confidence"]==48]

### Salary exploration

In [ ]:
df_year = job_hits[job_hits["salary_period"] == "year"][["salary_min", "salary_max"]].copy()
df_month = job_hits[job_hits["salary_period"] == "month"][["salary_min", "salary_max"]].copy()
df_day = job_hits[job_hits["salary_period"] == "day"][["salary_min", "salary_max"]].copy()
df_hour = job_hits[job_hits["salary_period"] == "hour"][["salary_min", "salary_max"]].copy()


In [ ]:
df_list = [(df_year,"year"), (df_month,"month"), (df_day,"day"), (df_hour,"hour")]

for df,time in df_list:
    long = df.melt(value_vars=["salary_min", "salary_max"], var_name="type", value_name="salary")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(long, x="salary", hue="type", bins=60, element="step", stat="count", common_norm=False, ax=ax)
    ax.set_title(f"Salary ({time}) — min vs max")
    plt.show()

### Canonical fingerprint vs job_id

##### how many fingerprint by job_id (split)

In [ ]:
job_hits[["external_job_id","fingerprint"]].groupby("external_job_id").nunique().value_counts()

In [ ]:
job_to_fp_n = (
    job_hits.dropna(subset=["fingerprint", "external_job_id"])
            .groupby("external_job_id")["fingerprint"]
            .nunique()
)
bad_job_ids = job_to_fp_n[job_to_fp_n > 1].sort_values(ascending=False)
bad_job_ids_list = bad_job_ids.index.tolist()
cols = ["external_job_id","fingerprint","title","company","location_raw","salary_raw","canonical_url"]
for bad_job_id in bad_job_ids_list:
    display(job_hits[job_hits["external_job_id"]==bad_job_id][cols].sort_values(["fingerprint","external_job_id"]).drop_duplicates())

the title changed

##### how many job_id by fingerprint (collisions)

In [ ]:
job_hits[["external_job_id","fingerprint"]].groupby("fingerprint").nunique().value_counts()

In [ ]:
fp_to_jobid_n = (
    job_hits.dropna(subset=["fingerprint", "external_job_id"])
            .groupby("fingerprint")["external_job_id"]
            .nunique()
)
bad_fps = fp_to_jobid_n[fp_to_jobid_n > 1].sort_values(ascending=False)
bad_fp_list = bad_fps.index.tolist()
cols = ["fingerprint","external_job_id","title","company","location_raw","salary_raw","canonical_url"]
for bad_fp in bad_fp_list:
    display(job_hits[job_hits["fingerprint"]==bad_fp][cols].sort_values(["fingerprint","external_job_id"]).drop_duplicates())

# Job Ads

In [ ]:
with db.get_connection(ws.db_path) as conn:
    job_ads = pd.read_sql_query(
        "SELECT * FROM job_ads",
        conn,
        index_col = 'id'
    )

job_ads

In [ ]:
job_ads.info()

## Location

In [ ]:
len(job_ads["city"].unique())

In [ ]:
len(job_ads["company"].unique())

In [ ]:
job_ads["city"].value_counts()

## Salary

In [ ]:
df_year = job_ads[job_ads["salary_period"] == "year"][["salary_min", "salary_max"]].copy()
df_month = job_ads[job_ads["salary_period"] == "month"][["salary_min", "salary_max"]].copy()
df_day = job_ads[job_ads["salary_period"] == "day"][["salary_min", "salary_max"]].copy()
df_hour = job_ads[job_ads["salary_period"] == "hour"][["salary_min", "salary_max"]].copy()

In [ ]:
df_list = [(df_year,"year"), (df_month,"month"), (df_day,"day"), (df_hour,"hour")]

for df,time in df_list:
    long = df.melt(value_vars=["salary_min", "salary_max"], var_name="type", value_name="salary")
    fig, ax = plt.subplots(figsize=(10, 4))
    sns.histplot(long, x="salary", hue="type", bins=60, element="step", stat="count", common_norm=False, ax=ax)
    ax.set_title(f"Salary ({time}) — min vs max")
    plt.show()